# Baseline-модель

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
sys.path.append(str(PROJECT_ROOT))

from sklearn.linear_model import LogisticRegression

from src.modeling import build_model_pipeline, evaluate_classifier, load_raw_data, split_data
from src.preprocessing import TARGET_COLUMN, clean_data

RANDOM_STATE = 42

Сначала загружаем исходный датасет, затем применяем только базовую очистку данных и делим выборку на train, validation и test. Это позволяет сразу увидеть, сколько строк осталось после очистки и как выглядит финальный объём данных для обучения без feature engineering.

In [2]:
raw_df = load_raw_data()
clean_df = clean_data(raw_df)
train_df, val_df, test_df = split_data(clean_df)

raw_df.shape, clean_df.shape, train_df.shape, val_df.shape, test_df.shape

((119390, 32), (86638, 30), (60646, 30), (12996, 30), (12996, 30))

In [3]:
x_train = train_df.drop(columns=[TARGET_COLUMN])
y_train = train_df[TARGET_COLUMN]
x_val = val_df.drop(columns=[TARGET_COLUMN])
y_val = val_df[TARGET_COLUMN]

## Baseline: Logistic Regression

Я выбрал `LogisticRegression` как baseline-модель. Это простая модель из коробки: если даже она показывает разумное качество, значит в данных есть устойчивый сигнал. Новые признаки здесь не создаются.

In [4]:
baseline_model = build_model_pipeline(
    train_df,
    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
)
baseline_model.fit(x_train, y_train)
baseline_metrics = evaluate_classifier(baseline_model, x_val, y_val)
baseline_metrics

{'accuracy': 0.796475838719606,
 'precision': 0.6856252434748734,
 'recall': 0.48916064480266813,
 'f1': 0.5709651257096513,
 'roc_auc': 0.8438380885032131}

Логистическая регрессия показывает, что задачу уже можно решать простой линейной моделью на очищенных данных. При этом качество ещё оставляет пространство для улучшения, поэтому дальше имеет смысл переходить к более гибким алгоритмам с feature engineering.